# Week 3: Data Preprocessing
- Handle missing values (decide whether to drop, impute, or flag).
- Convert categorical fields to numeric (encoding).
- Normalize numerical features if needed.
- Create train/test split: use the most recent month (2 months) of available data as the test set,
and the X months immediately preceding it as the training set. X is not fixed —
treat the training window length as a tunable choice and experiment to determine
the optimal value of X.
- Deliverable: 02_preprocessing.ipynb + cleaned CSV.


In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


### Reloading and filtering a new CSV of complete data files

Since our group initially loaded around 12 months of data, we will be expanding the scope of possible data that could be used to all the files we have access to.

In [3]:
import os
import glob
import re
import pandas as pd

def load_csv_files_from_folder(folder_path):
    """
    Loads all CSV files from the given folder into a list of Pandas DataFrames.
    Each DataFrame is paired with its filename and includes a 'FileDate' column
    extracted from the filename and converted to a pandas datetime object.
    Handles both YYYYMMDD (8 digits) and YYYYMM (6 digits) date formats.
    """
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    # Find all CSV files in the folder
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

    if not csv_files:
        print("No CSV files found in the folder.")
        return []

    dataframes = []
    for file_path in csv_files:
        filename = os.path.basename(file_path)
        try:
            df = pd.read_csv(file_path)

            # Extract either 8 consecutive digits or 6 consecutive digits representing a date
            date_match = re.search(r'\d{8}|\d{6}', filename)
            if date_match:
                date_str = date_match.group(0)
                if len(date_str) == 8:
                    file_date = pd.to_datetime(date_str, format='%Y%m%d')
                else:  # len(date_str) == 6
                    file_date = pd.to_datetime(date_str, format='%Y%m')
            else:
                file_date = pd.NaT

            # Create the 'FileDate' column (YYYYMM) with datetime type
            df['FileDate'] = file_date

            dataframes.append((filename, df))
            print(f"Loaded: {filename} ({len(df)} rows) with FileDate: {file_date.strftime('%Y-%m-%d') if not pd.isnull(file_date) else 'NaT'}")
        except Exception as e:
            print(f"Error loading {file_path}: {e}")

    return dataframes

In [4]:
# Load 6 months of data
data_list = load_csv_files_from_folder("/content/drive/MyDrive/IDX Data Science/Raw Data/sold")

Loaded: CRMLSSold202408.csv (24558 rows) with FileDate: 2024-08-01
Loaded: CRMLSSold202403_filled.csv (23276 rows) with FileDate: 2024-03-01
Loaded: CRMLSSold202603.csv (23372 rows) with FileDate: 2026-03-01
Loaded: CRMLSSold202508.csv (22972 rows) with FileDate: 2025-08-01
Loaded: CRMLSSold202409.csv (21267 rows) with FileDate: 2024-09-01
Loaded: CRMLSSold202507.csv (23646 rows) with FileDate: 2025-07-01
Loaded: CRMLSSold202405_filled.csv (26487 rows) with FileDate: 2024-05-01
Loaded: CRMLSSold202505.csv (23154 rows) with FileDate: 2025-05-01
Loaded: CRMLSSold202604.csv (24261 rows) with FileDate: 2026-04-01
Loaded: CRMLSSold202511.csv (19088 rows) with FileDate: 2025-11-01
Loaded: CRMLSSold202504.csv (23262 rows) with FileDate: 2025-04-01
Loaded: CRMLSSold202512.csv (20538 rows) with FileDate: 2025-12-01
Loaded: CRMLSSold202501_filled.csv (18738 rows) with FileDate: 2025-01-01
Loaded: CRMLSSold202402.csv (19925 rows) with FileDate: 2024-02-01
Loaded: CRMLSSold202401_filled.csv (17958

/tmp/ipykernel_846/2783330945.py:27: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Loaded: CRMLSSold202506.csv (22883 rows) with FileDate: 2025-06-01
Loaded: CRMLSSold202412.csv (20241 rows) with FileDate: 2024-12-01


/tmp/ipykernel_846/2783330945.py:27: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Loaded: CRMLSSold202601.csv (16487 rows) with FileDate: 2026-01-01
Loaded: CRMLSSold202602.csv (19010 rows) with FileDate: 2026-02-01


/tmp/ipykernel_846/2783330945.py:27: DtypeWarning: Columns (2,36,39,56,74) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Loaded: CRMLSSold202404_filled.csv (24640 rows) with FileDate: 2024-04-01
Loaded: CRMLSSold202407_filled.csv (26240 rows) with FileDate: 2024-07-01
Loaded: CRMLSSold202510.csv (23233 rows) with FileDate: 2025-10-01
Loaded: CRMLSSold202406_filled.csv (24328 rows) with FileDate: 2024-06-01
Loaded: CRMLSSold202503.csv (21445 rows) with FileDate: 2025-03-01
Loaded: CRMLSSold202502.csv (18702 rows) with FileDate: 2025-02-01
Loaded: CRMLSSold202410.csv (23274 rows) with FileDate: 2024-10-01


In [5]:
# Combine into dataframe
combined = pd.concat([df for __,df in data_list], ignore_index=True)
#combined.head(10)

In [6]:
# Filtering Data
print(f"Number of Rows before Restriction: {len(combined)}")

#Restrict Property Type to Residential and SubType to Single-Family Residence
residential = combined[combined['PropertyType'] == 'Residential']
print(f"Number of Rows after filtering by PropertyType: {len(residential)}")

#Restrict Property SubType to Single-Family Residence
filtered = residential[residential['PropertySubType'] == 'SingleFamilyResidence']
print(f"Number of Rows after full Restriction: {len(filtered)}")

filtered_data = filtered.to_csv('/content/drive/MyDrive/IDX Data Science/filtered_all.csv', index=False, encoding='utf-8')



Number of Rows before Restriction: 615707
Number of Rows after filtering by PropertyType: 414054
Number of Rows after full Restriction: 309735


### Inspecting Mixed-type columns

In [ ]:
print("Unique values in OriginatingSystemName (including NaNs):")
print(filtered.iloc[:, 78].unique())

print("\nValue counts including nulls:")
print(filtered.iloc[:, 78].value_counts(dropna=False))

print("\nNumber of null values:")
print(filtered.iloc[:, 78].isnull().sum())

In [ ]:
print("Unique values in OriginatingSystemSubName (including NaNs):")
print(filtered.iloc[:, 79].unique())

print("\nValue counts including nulls:")
print(filtered.iloc[:, 79].value_counts(dropna=False))

print("\nNumber of null values:")
print(filtered.iloc[:, 79].isnull().sum())

## Missing Value Analysis

In [10]:
filtered_cols = list(filtered.columns)
filtered_cols_types = list(filtered.dtypes)
filtered_missing_count = [filtered[col].isnull().sum() for col in filtered_cols]
filtered_missing_prop = [filtered[col].isnull().mean() for col in filtered_cols]
filtered_missing = pd.DataFrame({
    'Column Name': filtered_cols,
    'Data Type': filtered_cols_types,
    'Missing Count': filtered_missing_count,
    'Missing Proportion': filtered_missing_prop,
})

#inspecting columns with a large proportion of missing values
filtered_missing_80_or_more = filtered_missing[filtered_missing['Missing Proportion'] > 0.8].sort_values('Missing Proportion', ascending=False)
filtered_missing_80_or_more

,Column Name,Data Type,Missing Count,Missing Proportion
32,AboveGradeFinishedArea,float64,309735,1.000000
30,FireplacesTotal,float64,309735,1.000000
35,TaxAnnualAmount,float64,309735,1.000000
60,BusinessType,object,309735,1.000000
55,ElementarySchoolDistrict,float64,309735,1.000000
51,TaxYear,float64,309735,1.000000
77,MiddleOrJuniorSchoolDistrict,float64,309735,1.000000
62,CoveredSpaces,float64,309735,1.000000
4,WaterfrontYN,object,309599,0.999561
59,BelowGradeFinishedArea,float64,307832,0.993856


In [11]:
# missing values for key columns

#columns I think may have useful predictors for close price
key_feature_cols = ['ClosePrice', 'LivingArea', 'BedroomsTotal',
                    'BathroomsTotalInteger', 'LotSizeSquareFeet',
                    'YearBuilt', 'TotalActualRent', 'StateOrProvince',
                    'RoomsTotal', 'Latitude', 'Longitude', 'Furnished']
filtered_missing[filtered_missing['Column Name'].isin(key_feature_cols)]

,Column Name,Data Type,Missing Count,Missing Proportion
11,ClosePrice,float64,2,0.000006
14,Latitude,float64,2712,0.008756
15,Longitude,float64,2712,0.008756
18,LivingArea,float64,166,0.000536
46,YearBuilt,float64,225,0.000726
49,BathroomsTotalInteger,float64,50,0.000161
53,BedroomsTotal,float64,0,0.000000
61,StateOrProvince,object,0,0.000000
76,LotSizeSquareFeet,float64,5327,0.017199


Since the proportion of missing values for these key columns are very low, I will drop missing values in each column when fitting my models.

## Train-Test Split

In [12]:
def train_test_split(df, train_months, test_months=2):
    """
    Splits the dataframe into a training set and a testing set based on the unique months in the 'FileDate' column.

    Parameters:
    - df (pd.DataFrame): The input filtered DataFrame containing a 'FileDate' datetime column.
    - train_months (int): The number of consecutive months directly preceding the test set to use for training.
    - test_months (int): The number of most recent consecutive months to use for testing.

    Returns:
    - tuple: (train_df, test_df) in that order.
    """
    # Get unique, non-null months sorted in descending order (most recent first)
    unique_months = sorted(df['FileDate'].dropna().unique(), reverse=True)

    if len(unique_months) < (test_months + train_months):
        print(f"Warning: Not enough unique months in data ({len(unique_months)}) to satisfy test_months ({test_months}) and train_months ({train_months}).")

    # Test months are the top most recent unique dates
    test_month_cuts = unique_months[:test_months]

    # Train months are the next adjacent unique dates directly preceding the test set
    train_month_cuts = unique_months[test_months : test_months + train_months]

    print(f"Test months selected: {[d.strftime('%Y-%m') for d in test_month_cuts]}")
    print(f"Train months selected: {[d.strftime('%Y-%m') for d in train_month_cuts]}")

    # Slice the original DataFrame using the determined months
    test_df = df[df['FileDate'].isin(test_month_cuts)].copy()
    train_df = df[df['FileDate'].isin(train_month_cuts)].copy()

    print(f"Training rows: {len(train_df)}, Testing rows: {len(test_df)}")

    return train_df, test_df

# Example usage with 6 train months and 2 test months:
# train_set, test_set = train_test_split(filtered, train_months=6, test_months=2)

In [13]:
train, test = train_test_split(filtered, train_months=12, test_months=2)

Test months selected: ['2026-04', '2026-03']
Train months selected: ['2026-02', '2026-01', '2025-12', '2025-11', '2025-10', '2025-09', '2025-08', '2025-07', '2025-06', '2025-05', '2025-04', '2025-03']
Training rows: 129664, Testing rows: 24052


Checking missing value count/prop in train and test sets

In [15]:
## Check missing values in the test set
test_cols = list(test.columns)
test_cols_types = list(test.dtypes)
test_missing_count = [test[col].isnull().sum() for col in test_cols]
test_missing_prop = [test[col].isnull().mean() for col in test_cols]

test_missing = pd.DataFrame({
    'Column Name': test_cols,
    'Data Type': test_cols_types,
    'Missing Count': test_missing_count,
    'Missing Proportion': test_missing_prop,
})

# Display columns with missing values in the test set, sorted by proportion
display(test_missing[test_missing['Missing Proportion'] > 0.8].sort_values('Missing Proportion', ascending=False))

,Column Name,Data Type,Missing Count,Missing Proportion
32,AboveGradeFinishedArea,float64,24052,1.000000
30,FireplacesTotal,float64,24052,1.000000
35,TaxAnnualAmount,float64,24052,1.000000
60,BusinessType,object,24052,1.000000
55,ElementarySchoolDistrict,float64,24052,1.000000
51,TaxYear,float64,24052,1.000000
82,lonfilled,object,24052,1.000000
80,BuyerAgencyCompensation,float64,24052,1.000000
81,latfilled,object,24052,1.000000
62,CoveredSpaces,float64,24052,1.000000


In [16]:
test_missing[test_missing['Column Name'].isin(key_feature_cols)]

,Column Name,Data Type,Missing Count,Missing Proportion
11,ClosePrice,float64,0,0.000000
14,Latitude,float64,153,0.006361
15,Longitude,float64,153,0.006361
18,LivingArea,float64,8,0.000333
46,YearBuilt,float64,16,0.000665
49,BathroomsTotalInteger,float64,1,0.000042
53,BedroomsTotal,float64,0,0.000000
61,StateOrProvince,object,0,0.000000
76,LotSizeSquareFeet,float64,426,0.017712


In [19]:
## Check missing values in the training set
train_cols = list(train.columns)
train_cols_types = list(train.dtypes)
train_missing_count = [train[col].isnull().sum() for col in train_cols]
train_missing_prop = [train[col].isnull().mean() for col in train_cols]

train_missing = pd.DataFrame({
    'Column Name': train_cols,
    'Data Type': train_cols_types,
    'Missing Count': train_missing_count,
    'Missing Proportion': train_missing_prop,
})

# Display columns with missing values in the test set, sorted by proportion
display(train_missing[train_missing['Missing Proportion'] > 0.8].sort_values('Missing Proportion', ascending=False))

,Column Name,Data Type,Missing Count,Missing Proportion
32,AboveGradeFinishedArea,float64,129664,1.000000
30,FireplacesTotal,float64,129664,1.000000
51,TaxYear,float64,129664,1.000000
35,TaxAnnualAmount,float64,129664,1.000000
82,lonfilled,object,129664,1.000000
62,CoveredSpaces,float64,129664,1.000000
60,BusinessType,object,129664,1.000000
55,ElementarySchoolDistrict,float64,129664,1.000000
81,latfilled,object,129664,1.000000
80,BuyerAgencyCompensation,float64,129664,1.000000


In [20]:
train_missing[train_missing['Column Name'].isin(key_feature_cols)]

,Column Name,Data Type,Missing Count,Missing Proportion
11,ClosePrice,float64,0,0.000000
14,Latitude,float64,67,0.000517
15,Longitude,float64,67,0.000517
18,LivingArea,float64,74,0.000571
46,YearBuilt,float64,88,0.000679
49,BathroomsTotalInteger,float64,16,0.000123
53,BedroomsTotal,float64,0,0.000000
61,StateOrProvince,object,0,0.000000
76,LotSizeSquareFeet,float64,2239,0.017268
